# 🔄 Cross-Validation, Tuning, Imbalance, Leakage, Pipeline
---
## Concepts (Like You're 10):
- **Cross-Validation:** Judge student by 5 exams, not 1!
- **Hyperparameter Tuning:** Find the perfect recipe by trying combos
- **Imbalanced Data:** 95 cats, 5 dogs → model says ALL cats → useless!
- **Data Leakage:** Peeking at test answers during training → cheating!
- **Pipeline:** Chain steps together → no cheating, clean code

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score,
    StratifiedKFold, RandomizedSearchCV, GridSearchCV, learning_curve)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter
warnings.filterwarnings('ignore')

# Load & Quick Clean
df = pd.read_csv('loan_default_dataset.csv')
for c in df.select_dtypes(include=[np.number]).columns:
    df[c].fillna(df[c].median(), inplace=True)
for c in df.select_dtypes(include=['object']).columns:
    df[c].fillna(df[c].mode()[0], inplace=True)
    if df[c].nunique() == 2:
        df[c] = LabelEncoder().fit_transform(df[c])
    else:
        df = pd.concat([df.drop(c, axis=1),
                        pd.get_dummies(df[c], prefix=c, drop_first=True)], axis=1)

X = df.drop('Default', axis=1)
y = df['Default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train),
                          columns=X_train.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test),
                         columns=X_test.columns, index=X_test.index)
print(f'✅ Data ready! Train: {X_train.shape}, Test: {X_test.shape}')

## 1️⃣ CROSS-VALIDATION (5-Fold Stratified)
**WHY:** Single train-test split = judging by ONE exam.
CV = 5 exams → more reliable!

**Stratified** = each fold has same class ratio (critical for imbalanced data!)

In [ ]:
print('🔄 5-Fold Stratified Cross-Validation')
print('=' * 60)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000,
        class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100,
        max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5,
        random_state=42, use_label_encoder=False,
        eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=5,
        random_state=42, verbose=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train_s, y_train,
                             cv=skf, scoring='f1', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'Mean F1': round(scores.mean(), 4),
        'Std': round(scores.std(), 4),
        'Fold Scores': scores.round(4)
    })
    print(f'  {name:25s} → F1: {scores.mean():.4f} (+/- {scores.std():.4f})')
    print(f'    Folds: {scores.round(4)}')

cv_df = pd.DataFrame(cv_results)
print(f'\n📊 Summary:')
print(cv_df[['Model', 'Mean F1', 'Std']])

# Visualize
plt.figure(figsize=(10, 5))
plt.bar(cv_df['Model'], cv_df['Mean F1'], yerr=cv_df['Std'],
        capsize=5, color='steelblue', edgecolor='black')
plt.ylabel('Mean F1 Score')
plt.title('5-Fold Stratified CV Results', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 2️⃣ HANDLE IMBALANCED DATA (SMOTE)
**WHY:** If 95% = No Default, model predicts ALL No → 95% accuracy but USELESS!

**SMOTE** = Creates SYNTHETIC minority samples (not just copies!)

⚠️ **CRITICAL:** Apply SMOTE ONLY on TRAINING data, NEVER on test!

In [ ]:
print('⚖️ IMBALANCED DATA HANDLING')
print('=' * 60)
print(f'Before SMOTE: {Counter(y_train)}')
ratio = Counter(y_train)[0] / max(Counter(y_train)[1], 1)
print(f'Imbalance ratio: {ratio:.2f}:1')

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_s, y_train)
print(f'\nAfter SMOTE:  {Counter(y_train_bal)}')
print(f'Training size: {len(X_train)} → {len(X_train_bal)}')

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
pd.Series(y_train).value_counts().plot(kind='bar', ax=ax1,
    color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax1.set_title('Before SMOTE', fontweight='bold')
ax1.set_xticklabels(['No Default', 'Default'], rotation=0)

pd.Series(y_train_bal).value_counts().plot(kind='bar', ax=ax2,
    color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax2.set_title('After SMOTE', fontweight='bold')
ax2.set_xticklabels(['No Default', 'Default'], rotation=0)
plt.tight_layout()
plt.show()

# Compare model with and without SMOTE
rf_no_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_no_smote.fit(X_train_s, y_train)
print(f'\nRF without SMOTE → F1: {f1_score(y_test, rf_no_smote.predict(X_test_s)):.4f}')

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_bal, y_train_bal)
print(f'RF with SMOTE    → F1: {f1_score(y_test, rf_smote.predict(X_test_s)):.4f}')

## 3️⃣ HYPERPARAMETER TUNING
**WHY:** Default settings are rarely optimal.
Tuning = Finding the perfect recipe!

- **GridSearch:** Try ALL combos (slow but thorough)
- **RandomSearch:** Try RANDOM combos (faster, often just as good)
- **Bayesian (Optuna):** Smart search, learns from previous tries

In [ ]:
print('⚙️ HYPERPARAMETER TUNING — Random Forest')
print('=' * 60)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_dist,
    n_iter=30,                    # Try 30 random combos
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train_bal, y_train_bal)

print(f'🏆 Best Parameters:')
for k, v in random_search.best_params_.items():
    print(f'   {k}: {v}')
print(f'\n🏆 Best CV F1: {random_search.best_score_:.4f}')

# Evaluate on test
best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test_s)
print(f'\n📊 Test Results:')
print(f'   Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'   F1 Score: {f1_score(y_test, y_pred):.4f}')

In [ ]:
print('⚙️ HYPERPARAMETER TUNING — XGBoost')
print('=' * 60)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False,
                  eval_metric='logloss', verbosity=0),
    xgb_params, n_iter=30,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1', random_state=42, n_jobs=-1, verbose=0
)

xgb_search.fit(X_train_bal, y_train_bal)

print(f'🏆 Best Parameters:')
for k, v in xgb_search.best_params_.items():
    print(f'   {k}: {v}')
print(f'\n🏆 Best CV F1: {xgb_search.best_score_:.4f}')

best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_s)
print(f'\n📊 Test F1: {f1_score(y_test, y_pred_xgb):.4f}')

## 4️⃣ DATA LEAKAGE — THE SILENT KILLER
**WHAT:** Training data accidentally contains info from test data.
**RESULT:** Model looks AMAZING in testing, FAILS in real world!

**Common Causes:**
1. Scaling BEFORE splitting ❌
2. Using future data to predict past ❌
3. Target info in features ❌

**Prevention:** ALWAYS split first → then preprocess → Use PIPELINES!

In [ ]:
print('⚠️ DATA LEAKAGE DEMO')
print('=' * 60)

# ❌ WRONG WAY (Data Leakage!)
print('\n❌ WRONG: Scale ALL data first, then split')
scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(X)  # LEAKAGE! Test info in training!
X_tr_wrong, X_te_wrong, y_tr, y_te = train_test_split(
    X_all_scaled, y, test_size=0.2, random_state=42)
rf_wrong = RandomForestClassifier(n_estimators=100, random_state=42)
rf_wrong.fit(X_tr_wrong, y_tr)
print(f'   F1 (leaked): {f1_score(y_te, rf_wrong.predict(X_te_wrong)):.4f}')
print('   ↑ Looks good but UNRELIABLE in real world!')

# ✅ RIGHT WAY (No Leakage)
print('\n✅ RIGHT: Split first, then scale')
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler_right = StandardScaler()
X_tr2_s = scaler_right.fit_transform(X_tr2)  # fit ONLY on train
X_te2_s = scaler_right.transform(X_te2)      # transform test
rf_right = RandomForestClassifier(n_estimators=100, random_state=42)
rf_right.fit(X_tr2_s, y_tr2)
print(f'   F1 (correct): {f1_score(y_te2, rf_right.predict(X_te2_s)):.4f}')
print('   ↑ This is the REAL performance!')

## 5️⃣ ML PIPELINE
**WHAT:** Chain preprocessing + model into ONE object.
**WHY:** Prevents data leakage, cleaner code, easier deployment!

```
Pipeline = [Scaler → Model] → One object to save & deploy!
```

In [ ]:
print('🔄 ML PIPELINE')
print('=' * 60)

# Create Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),           # Step 1: Scale
    ('classifier', RandomForestClassifier(  # Step 2: Classify
        n_estimators=200, max_depth=10,
        class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])

# Use ORIGINAL unscaled data — pipeline handles scaling!
pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)

print(f'Pipeline Results:')
print(f'   Accuracy: {accuracy_score(y_test, y_pred_pipe):.4f}')
print(f'   F1 Score: {f1_score(y_test, y_pred_pipe):.4f}')
print(f'   AUC:      {roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]):.4f}')

# Cross-validate the pipeline
pipe_scores = cross_val_score(pipeline, X_train, y_train,
                               cv=5, scoring='f1', n_jobs=-1)
print(f'\n   CV F1: {pipe_scores.mean():.4f} (+/- {pipe_scores.std():.4f})')

print('\n💡 Pipeline Benefits:')
print('   1. No data leakage (scaling inside pipeline)')
print('   2. One object to save/deploy')
print('   3. Consistent preprocessing')
print('   4. Easy to swap models')

In [ ]:
# Pipeline with Tuning
print('⚙️ Pipeline + Hyperparameter Tuning')
print('=' * 60)

pipe_tuned = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(class_weight='balanced',
                                    random_state=42, n_jobs=-1))
])

# Note: Use clf__ prefix for pipeline parameters!
pipe_params = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [5, 10, None],
    'clf__min_samples_split': [5, 10]
}

pipe_search = RandomizedSearchCV(
    pipe_tuned, pipe_params, n_iter=10,
    cv=3, scoring='f1', random_state=42, n_jobs=-1
)
pipe_search.fit(X_train, y_train)  # Use raw data!

print(f'Best params: {pipe_search.best_params_}')
print(f'Best CV F1: {pipe_search.best_score_:.4f}')
print(f'Test F1: {f1_score(y_test, pipe_search.predict(X_test)):.4f}')

In [ ]:
# Learning Curves
print('📈 LEARNING CURVES')
print('=' * 60)
print('Diagnose: Need more DATA or more COMPLEXITY?')

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=100, max_depth=10,
                           class_weight='balanced', random_state=42, n_jobs=-1),
    X_train_bal, y_train_bal,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='f1', n_jobs=-1
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-',
         label='Training', color='blue')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-',
         label='Validation', color='red')
plt.fill_between(train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.1, color='blue')
plt.fill_between(train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.1, color='red')
plt.xlabel('Training Set Size')
plt.ylabel('F1 Score')
plt.title('Learning Curve', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print('\n📖 How to Read:')
print('  Both HIGH & close → Good model ✅')
print('  Train HIGH, Val LOW → Overfitting ⚠️ (simplify)')
print('  Both LOW → Underfitting ⚠️ (add complexity)')